# Module 2 — TF-IDF Logistic Regression

Train and save the sparse-text component used by the final ensemble. Word features capture terms and short phrases; character features help with Romanized Bangla spelling variations. Both learn from training text only.

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODEL_DIR, RANDOM_SEED
from src.dataset_utils import load_fixed_data_splits
from src.evaluation import calculate_metrics, save_evaluation_outputs, update_metrics_file

train_data, validation_data, test_data = load_fixed_data_splits()

## Build TF-IDF features and train Logistic Regression

In [ ]:
vectorizer = FeatureUnion([
    ('word', TfidfVectorizer(
        lowercase=False, token_pattern=r'(?u)\S+',
        min_df=2, ngram_range=(1, 2), sublinear_tf=True,
    )),
    ('character', TfidfVectorizer(
        lowercase=False, analyzer='char_wb',
        min_df=3, ngram_range=(3, 5), sublinear_tf=True,
        max_features=100000,
    )),
])
train_features = vectorizer.fit_transform(train_data['processed_text'])
validation_features = vectorizer.transform(validation_data['processed_text'])
test_features = vectorizer.transform(test_data['processed_text'])

classifier = LogisticRegression(
    max_iter=500,
    solver='lbfgs',
    random_state=RANDOM_SEED,
)
classifier.fit(train_features, train_data['label'])

## Evaluate and save the ensemble component

In [ ]:
validation_predictions = classifier.predict(validation_features)
validation_f1 = calculate_metrics(validation_data['label'], validation_predictions)['Macro F1']
test_predictions = classifier.predict(test_features)
result = save_evaluation_outputs(
    experiment_id='M2.4',
    experiment_name='TF-IDF + Logistic Regression',
    family='Discriminative',
    test_data=test_data,
    predictions=test_predictions.tolist(),
)
result['Representation'] = 'Word + Character TF-IDF'
result['Validation Macro F1'] = validation_f1
selected_metrics = update_metrics_file(pd.DataFrame([result]))

joblib.dump({
    'name': 'TF-IDF + Logistic Regression',
    'vectorizer': vectorizer,
    'classifier': classifier,
    'classes': classifier.classes_.tolist(),
    'validation_macro_f1': validation_f1,
}, MODEL_DIR / 'tfidf_logistic.pkl')
selected_metrics[selected_metrics['Experiment ID'] == 'M2.4']